# M1.2/M1.3 — Model dasar (kontrol) dan adaptor sebagai LoRA-runtime
Menghasilkan dua berkas GGUF **tanpa menggabungkan** apa pun dan **tanpa mengubah baseline**:
1. `qwen2.5-3b-instruct-q8_0.gguf` — model dasar tanpa adaptor (kontrol: apa yang dilakukan/dirusak fine-tune?)
2. `celcius-v1-lora-f16.gguf` — adaptor `celcius-v1` dalam format GGUF-LoRA (dimuat llama-server dengan `--lora`, skala bisa diubah)

Runtime **CPU** cukup (tidak ada penggabungan model). Siapkan `adapter.zip` (folder adaptor di-zip). **Belum dijalankan di Colab**; bila ada sel yang error, kirim teksnya.

In [ ]:
!pip install -q gguf sentencepiece protobuf huggingface_hub

## 1. Unduh model dasar dari Hugging Face

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen2.5-3B-Instruct', local_dir='/content/base', allow_patterns=['*.json','*.safetensors','*.txt'])
!ls -la /content/base | head -30

## 2. llama.cpp (versi yang sama dengan llama-server Anda: build b12068)
Konverter dan runtime sebaiknya satu versi. Bila tag tidak ditemukan, sel akan memakai versi terbaru.

In [ ]:
!rm -rf llama.cpp
!git clone -q --depth 1 --branch b12068 https://github.com/ggml-org/llama.cpp || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!ls llama.cpp | head -5

## 3. Model dasar -> GGUF Q8_0 (kontrol)

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py /content/base --outfile /content/qwen2.5-3b-instruct-q8_0.gguf --outtype q8_0
!ls -lh /content/*.gguf

## 4. Unggah adaptor dan ubah ke GGUF-LoRA

In [ ]:
from google.colab import files
up = files.upload()          # pilih adapter.zip
print(list(up))

In [ ]:
import glob, os, json
!unzip -o -q /content/adapter.zip -d /content/adapter_raw
hits = glob.glob('/content/adapter_raw/**/adapter_config.json', recursive=True)
assert hits, 'adapter_config.json tidak ditemukan di dalam zip'
ADAPTER = os.path.dirname(hits[0])
cfg = json.load(open(ADAPTER + '/adapter_config.json'))
print('adapter:', ADAPTER)
print('model dasar menurut adaptor:', cfg['base_model_name_or_path'], '| r =', cfg['r'], '| alpha =', cfg['lora_alpha'])
assert cfg['base_model_name_or_path'] == 'Qwen/Qwen2.5-3B-Instruct', 'model dasar tidak sama dengan yang diunduh di langkah 1'

In [ ]:
# Cetak opsi konverter agar jelas (opsi bisa berbeda antar versi):
!python llama.cpp/convert_lora_to_gguf.py --help | head -40

In [ ]:
!python llama.cpp/convert_lora_to_gguf.py --base /content/base --outtype f16 --outfile /content/celcius-v1-lora-f16.gguf {ADAPTER}
!ls -lh /content/*.gguf

## 5. Ambil hasilnya
Unduh langsung, atau simpan ke Google Drive bila unduhan terputus. **Jangan menimpa** `celcius-q8_0.gguf` Anda; simpan keduanya dengan nama baru di `%APPDATA%\Celcius\models\` (atau folder lain).

In [ ]:
from google.colab import files
files.download('/content/qwen2.5-3b-instruct-q8_0.gguf')
files.download('/content/celcius-v1-lora-f16.gguf')

In [ ]:
# (alternatif) simpan ke Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/qwen2.5-3b-instruct-q8_0.gguf /content/celcius-v1-lora-f16.gguf /content/drive/MyDrive/

## 6. Menguji di komputer Anda (tutup Celcius dulu)
Di PowerShell, folder proyek:
```
llama\llama-server.exe -m <qwen2.5-3b-instruct-q8_0.gguf> --port 8099 -c 2048 -ngl 0 -np 1
node eval\run-quality.js --base http://127.0.0.1:8099 --label base-q8
```
Lalu dengan adaptor (skala 1.0, lalu 0.5, 0.75):
```
llama\llama-server.exe -m <qwen2.5-3b-instruct-q8_0.gguf> --lora-scaled <celcius-v1-lora-f16.gguf> 1.0 --port 8099 -c 2048 -ngl 0 -np 1
node eval\run-quality.js --base http://127.0.0.1:8099 --label lora-runtime-1.0
```
Bila opsi `--lora-scaled` ditolak, jalankan `llama\llama-server.exe --help` dan kirim bagian yang menyebut `lora`.